# Lesson 7 — Gravitational-Wave Waveform Models

**Gravitational-Wave Data Analysis with Python**  
LVK Python Course — Module 7

> *"To detect a signal you must first know what you are looking for — and that knowledge is encoded in a waveform model."*

---

## Table of Contents

1. [Introduction: Why Waveform Models Matter](#1-introduction)
2. [Post-Newtonian (PN) Approximants](#2-post-newtonian)
   - 2.1 The PN expansion in a nutshell
   - 2.2 TaylorF2: frequency-domain PN
   - 2.3 TaylorT4: time-domain PN (ODE integration)
3. [Effective One-Body (EOB) Framework](#3-eob)
   - 3.1 Key ideas of EOB
   - 3.2 SEOBNRv4 in PyCBC
4. [Phenomenological Approximants (IMRPhenom)](#4-imrphenom)
   - 4.1 Construction philosophy
   - 4.2 IMRPhenomD, IMRPhenomPv2, IMRPhenomXP
5. [Numerical-Relativity Surrogates](#5-surrogates)
   - 5.1 From NR grids to fast surrogates
   - 5.2 NRHybSur3dq8 in Python
6. [Physical Effects on Waveforms](#6-physics)
   - 6.1 Spin precession
   - 6.2 Orbital eccentricity
   - 6.3 Tidal deformability Λ (BNS)
7. [Comparing Waveform Families: Mismatch & Cost](#7-comparison)
   - 7.1 The mismatch metric
   - 7.2 Systematic mismatch survey
   - 7.3 Computational cost benchmarks
8. [Signal Injection & Recovery Analysis](#8-injection)
   - 8.1 Injecting into real noise
   - 8.2 Matched-filter recovery with cross-approximant templates
   - 8.3 Efficiency curves and fitting factor
9. [Additional Topics](#9-additional)
   - 9.1 Waveform models for LISA and Einstein Telescope
   - 9.2 Machine-learning surrogate models
10. [Student Exercises](#10-exercises)
11. [References](#11-references)


---
## 1. Introduction: Why Waveform Models Matter <a id="1-introduction"></a>

Gravitational-wave (GW) searches and parameter estimation both rely on **template waveforms** — theoretical predictions of the strain $h(t)$ (or $\tilde h(f)$) emitted by a compact binary coalescence (CBC).

A CBC consists of three qualitatively different phases:

| Phase | Duration | Dominant physics |
|---|---|---|
| **Inspiral** | Days–minutes | Post-Newtonian perturbation theory, tidal coupling |
| **Merger** | ~ms | Fully nonlinear GR (requires NR) |
| **Ring-down** | ~10 ms | Black-hole quasi-normal modes |

No single analytic framework covers all three phases, so the community developed a *hierarchy* of approximants — each optimised for accuracy, speed, or physical completeness in a different regime.

### Why the choice of approximant matters

* **Detection efficiency**: a mismatch of $\mathcal{M}_{\rm mm} \gtrsim 3\%$ causes a fractional SNR loss and a corresponding volume loss.
* **Parameter bias**: if the template family does not capture spin-precession or tidal effects, recovered parameters are biased.
* **Computational cost**: EOB/NR-surrogate waveforms can be 10–1000× slower than PN/phenom waveforms, constraining their use in large template banks.

The goal of this lesson is to give you a practical, hands-on understanding of all four major approximant families and the physical effects they capture.


In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import time, warnings
warnings.filterwarnings("ignore")

# PyCBC — the backbone library for waveforms and data
try:
    import pycbc
    from pycbc.waveform import get_fd_waveform, get_td_waveform
    from pycbc.filter import match, overlap, sigma
    from pycbc.psd import aLIGOZeroDetHighPower
    from pycbc.types import FrequencySeries, TimeSeries
    from pycbc.frame import read_frame
    PYCBC_AVAILABLE = True
    print(f"PyCBC {pycbc.__version__} available ✓")
except ImportError:
    PYCBC_AVAILABLE = False
    print("PyCBC not found — install with: pip install pycbc")

# GWpy — for frame/channel reading helpers
try:
    from gwpy.timeseries import TimeSeries as GWpyTS
    GWPY_AVAILABLE = True
    print("GWpy available ✓")
except ImportError:
    GWPY_AVAILABLE = False
    print("GWpy not found  (optional)")

# LALSimulation — used under the hood by PyCBC; nice to know the version
try:
    import lal, lalsimulation as lalsim
    print(f"LALSuite {lal.VCSInfo.version} available ✓")
    LALSIM_AVAILABLE = True
except ImportError:
    LALSIM_AVAILABLE = False
    print("LALSuite not found (optional)")

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
print("\nSetup complete ✓")


---
## 2. Post-Newtonian (PN) Approximants <a id="2-post-newtonian"></a>

### 2.1 The PN Expansion in a Nutshell

Post-Newtonian theory expands the Einstein field equations in powers of the small parameter

$$v/c \sim (G M \pi f / c^3)^{1/3} \ll 1$$

where $v$ is the orbital velocity, $M = m_1 + m_2$ the total mass, and $f$ the GW frequency.

The **gravitational-wave phase** to 3.5PN order is

$$\Psi_{\rm PN}(f) = 2\pi f t_c - \phi_c - \frac{\pi}{4} + \frac{3}{128\,\eta\, v^5} \sum_{k=0}^{7} \left(\alpha_k + \beta_k \ln v\right) v^k$$

with $v \equiv (\pi M f)^{1/3}$, $\eta = m_1 m_2/M^2$ the symmetric mass ratio, and coefficients $\alpha_k, \beta_k$ that depend on mass ratio, spins, and tidal parameters.

Key waveform families in this class:

| Approximant | Domain | PN order | Notes |
|---|---|---|---|
| **TaylorF2** | Frequency | 3.5PN phase | Stationary-phase approximation; fastest |
| **TaylorT1** | Time | 3.5PN | ODE on energy/flux |
| **TaylorT2** | Time | 3.5PN | Implicit ODE |
| **TaylorT4** | Time | 3.5PN | Explicit ODE; widely used |
| **SpinTaylorT4** | Time | 3.5PN + spin | Spin precession via Euler equations |

PN approximants are **not reliable near merger** — they diverge as $v \to 1$. They are used mainly for early inspiral ($f \lesssim 100$ Hz for stellar-mass BBH).

### 2.2 TaylorF2 — Frequency-Domain PN


In [ ]:
# ── 2.2  TaylorF2 waveform ──────────────────────────────────────────────────
# System parameters — GW150914-like BBH
m1, m2 = 36.0, 29.0          # component masses [M_sun]
spin1z = spin2z = 0.0         # aligned spins
f_lower = 20.0                # low-frequency cutoff [Hz]
delta_f = 0.0625              # frequency resolution [Hz]

if PYCBC_AVAILABLE:
    hp_f2, hc_f2 = get_fd_waveform(
        approximant="TaylorF2",
        mass1=m1, mass2=m2,
        spin1z=spin1z, spin2z=spin2z,
        delta_f=delta_f,
        f_lower=f_lower,
        distance=410.0,           # [Mpc]
    )
    freqs_f2 = hp_f2.sample_frequencies.numpy()
    amp_f2   = np.abs(hp_f2.numpy())

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Amplitude spectrum
    ax = axes[0]
    ax.loglog(freqs_f2[freqs_f2 > f_lower], amp_f2[freqs_f2 > f_lower],
              color="royalblue", lw=1.5, label="TaylorF2")
    # Newtonian leading-order: h ~ f^{-7/6}
    fref = 50.0
    idx_ref = np.argmin(np.abs(freqs_f2 - fref))
    f_plot = freqs_f2[(freqs_f2 > 25) & (freqs_f2 < 500)]
    h_newton = amp_f2[idx_ref] * (f_plot / fref)**(-7/6)
    ax.loglog(f_plot, h_newton, "k--", lw=1, label=r"Newtonian $f^{-7/6}$")
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"$|\tilde{h}(f)|$")
    ax.set_title("TaylorF2 Amplitude Spectrum")
    ax.legend()
    ax.set_xlim(20, 1024)

    # Phase derivative → instantaneous frequency evolution
    ax = axes[1]
    ph = np.unwrap(np.angle(hp_f2.numpy()))
    # Only show above f_lower
    mask = freqs_f2 > f_lower
    ax.plot(freqs_f2[mask], ph[mask] / (2*np.pi), color="darkorange", lw=1.5)
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"Phase / $2\pi$ [cycles]")
    ax.set_title("TaylorF2 Accumulated Phase")
    ax.set_xscale("log")
    plt.tight_layout()
    plt.savefig("/tmp/taylorf2.png", dpi=120)
    plt.show()
    print(f"TaylorF2 waveform length: {len(hp_f2)} frequency bins")
    print(f"Frequency range: {freqs_f2[freqs_f2>0].min():.3f} – {freqs_f2.max():.1f} Hz")
else:
    print("PyCBC not available — skipping waveform generation.")


### 2.3 TaylorT4 — Time-Domain ODE Integration

TaylorT4 integrates the coupled ODEs

$$\frac{d\omega}{dt} = \frac{32}{5} \frac{\eta}{M} \omega^{11/3} \sum_k c_k \omega^{k/3}, \qquad \frac{d\phi}{dt} = \omega$$

starting at $f_{\rm lower}$ and stopping when $\omega$ reaches the innermost stable circular orbit (ISCO).  The strain is then

$$h_+(t) \propto (G\mathcal{M}_c)^{5/3} \omega^{2/3}(t) \cos[\phi(t)].$$


In [ ]:
# ── 2.3  TaylorT4 time-domain waveform ─────────────────────────────────────
delta_t = 1.0 / 4096.0  # sample rate = 4096 Hz

if PYCBC_AVAILABLE:
    hp_t4, hc_t4 = get_td_waveform(
        approximant="TaylorT4",
        mass1=m1, mass2=m2,
        spin1z=0.0, spin2z=0.0,
        delta_t=delta_t,
        f_lower=f_lower,
        distance=410.0,
    )
    t_t4 = hp_t4.sample_times.numpy()

    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

    # Full waveform
    axes[0].plot(t_t4, hp_t4.numpy(), color="steelblue", lw=0.7)
    axes[0].set_ylabel(r"$h_+(t)$")
    axes[0].set_title("TaylorT4 Time-Domain Waveform (GW150914-like)")

    # Zoom on merger
    t_merger_approx = t_t4[-1]
    mask_zoom = t_t4 > (t_merger_approx - 0.5)
    axes[1].plot(t_t4[mask_zoom], hp_t4.numpy()[mask_zoom],
                 color="steelblue", lw=1.2, label="TaylorT4")
    axes[1].set_xlabel("GPS time offset [s]  (merger at right end)")
    axes[1].set_ylabel(r"$h_+(t)$")
    axes[1].set_title("Last 0.5 s of Inspiral (TaylorT4 diverges before merger)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    print(f"TaylorT4 duration: {t_t4[-1]-t_t4[0]:.1f} s  ({len(hp_t4)} samples)")
else:
    print("PyCBC not available — skipping.")


---
## 3. Effective One-Body (EOB) Framework <a id="3-eob"></a>

### 3.1 Key Ideas

Introduced by Buonanno & Damour (1999), the **Effective One-Body** (EOB) approach maps the two-body problem in GR onto the motion of a single test particle in a *deformed* Schwarzschild (or Kerr) geometry:

$$H_{\rm EOB} = M c^2 \sqrt{1 + 2\nu\left(\hat{H}_{\rm eff} - 1\right)}$$

where $\nu \equiv \mu/M$ is the symmetric mass ratio, and $\hat{H}_{\rm eff}$ is the effective Hamiltonian of the test particle.

Key advantages over pure PN:

1. **Resums divergent PN series** — the EOB Hamiltonian, radiation-reaction force, and waveform modes are each independently resummed.
2. **Includes merger and ring-down** — the EOB dynamics is evolved through the light-ring, then matched onto quasi-normal modes.
3. **Calibrated against NR** — free parameters (e.g. pseudo-4PN coefficient $a_6$) are fit to NR simulations.

Major EOB families:

| Family | Key features | Speed |
|---|---|---|
| SEOBNRv4 | Aligned spin, calibrated to ~141 NR sims | ~0.5–2 s per waveform |
| SEOBNRv5 | Improved HOM, BNS tides | ~1–5 s |
| TEOBResumS | BNS tides, eccentric | ~1–10 s |
| SEOBNRE | Eccentric BBH | ~10 s |

### 3.2 SEOBNRv4 in PyCBC


In [ ]:
# ── 3.2  SEOBNRv4 frequency-domain waveform ────────────────────────────────
if PYCBC_AVAILABLE:
    t0 = time.time()
    hp_seob, hc_seob = get_fd_waveform(
        approximant="SEOBNRv4_ROM",   # ROM = reduced-order model for speed
        mass1=m1, mass2=m2,
        spin1z=0.0, spin2z=0.0,
        delta_f=delta_f,
        f_lower=f_lower,
        distance=410.0,
    )
    t_seob = time.time() - t0
    print(f"SEOBNRv4_ROM generated in {t_seob:.3f} s")

    # Also generate TaylorF2 for direct comparison
    t0 = time.time()
    hp_f2b, _ = get_fd_waveform(
        approximant="TaylorF2",
        mass1=m1, mass2=m2,
        spin1z=0.0, spin2z=0.0,
        delta_f=delta_f,
        f_lower=f_lower,
        distance=410.0,
    )
    t_pn = time.time() - t0
    print(f"TaylorF2 generated in {t_pn:.3f} s")

    # Match the lengths
    n = min(len(hp_seob), len(hp_f2b))
    freqs = hp_seob.sample_frequencies.numpy()[:n]

    fig, axes = plt.subplots(2, 1, figsize=(13, 7))

    # Amplitude comparison
    ax = axes[0]
    ax.loglog(freqs[freqs > f_lower],
              np.abs(hp_f2b.numpy()[:n])[freqs > f_lower],
              color="royalblue", lw=2, label="TaylorF2 (PN)")
    ax.loglog(freqs[freqs > f_lower],
              np.abs(hp_seob.numpy()[:n])[freqs > f_lower],
              color="crimson", lw=1.5, ls="--", label="SEOBNRv4_ROM (EOB)")
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"$|\tilde{h}(f)|$  [strain / Hz$^{1/2}$]")
    ax.set_title("Amplitude: TaylorF2 vs SEOBNRv4_ROM")
    ax.legend()
    ax.set_xlim(20, 1024)

    # Phase difference (after aligning at f_lower)
    ax = axes[1]
    mask = (freqs > f_lower) & (freqs < 500)
    ph_pn   = np.unwrap(np.angle(hp_f2b.numpy()[:n][mask]))
    ph_seob = np.unwrap(np.angle(hp_seob.numpy()[:n][mask]))
    # Align at first sample
    d_ph = (ph_seob - ph_pn) - (ph_seob[0] - ph_pn[0])
    ax.plot(freqs[mask], d_ph / (2*np.pi),
            color="purple", lw=1.5)
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"$\Delta\Psi / (2\pi)$ [cycles]")
    ax.set_title("Phase Difference: EOB minus PN (higher-order EOB corrections)")
    ax.set_xscale("log")
    plt.tight_layout()
    plt.show()
else:
    print("PyCBC not available — skipping EOB comparison.")


---
## 4. Phenomenological Approximants (IMRPhenom) <a id="4-imrphenom"></a>

### 4.1 Construction Philosophy

The **IMRPhenom** family (Ajith et al. 2007 → Pratten et al. 2021) is built by:

1. **Constructing a frequency-domain ansatz** with distinct inspiral, intermediate, and merger-ring-down regions, each parameterised by smooth functions of $(\mathcal{M}_c, \eta, \chi)$.
2. **Fitting the free parameters** to a large bank of EOB+NR waveforms.
3. **Analytic closure** — once fitted, evaluating a waveform is just computing a few rational functions of frequency.

This makes IMRPhenom waveforms **extremely fast** (< 1 ms) while achieving ~99% match with NR.

Generation line:

$$\tilde{h}(f) = A(f)\, e^{i\Psi(f)}$$

with $A(f)$ and $\Psi(f)$ piecewise-smooth functions.

| Approximant | Spins | HOM | Precession | Speed |
|---|---|---|---|---|
| IMRPhenomA | No | No | No | ~0.1 ms |
| IMRPhenomD | Aligned | No | No | ~0.5 ms |
| IMRPhenomPv2 | Aligned+twist | No | Yes | ~2 ms |
| IMRPhenomHM | Aligned | Yes | No | ~1 ms |
| IMRPhenomXP | Aligned+twist | No | Yes | ~3 ms |
| IMRPhenomXPHM | Aligned+twist | Yes | Yes | ~5 ms |

### 4.2 Generating IMRPhenomD, IMRPhenomPv2, and IMRPhenomXP


In [ ]:
# ── 4.2  IMRPhenom family — generate and compare ───────────────────────────
approx_list = [
    ("TaylorF2",     dict(spin1z=0.0, spin2z=0.0)),
    ("IMRPhenomD",   dict(spin1z=0.3, spin2z=-0.1)),
    ("IMRPhenomPv2", dict(spin1z=0.3, spin2z=-0.1,
                          spin1x=0.2, spin1y=0.0,
                          spin2x=0.0, spin2y=0.0,
                          inclination=0.4)),
]

colors  = ["royalblue", "crimson", "seagreen"]
results = {}

if PYCBC_AVAILABLE:
    fig, axes = plt.subplots(2, 1, figsize=(13, 7))

    for (name, kwargs), col in zip(approx_list, colors):
        t0 = time.time()
        try:
            hp, _ = get_fd_waveform(
                approximant=name,
                mass1=m1, mass2=m2,
                delta_f=delta_f,
                f_lower=f_lower,
                distance=410.0,
                **kwargs,
            )
            elapsed = time.time() - t0
            results[name] = {"hp": hp, "time": elapsed}
            freqs = hp.sample_frequencies.numpy()
            amp   = np.abs(hp.numpy())
            mask  = freqs > f_lower

            axes[0].loglog(freqs[mask], amp[mask], color=col, lw=1.8,
                           label=f"{name} ({elapsed*1e3:.1f} ms)")
            ph = np.unwrap(np.angle(hp.numpy()))
            axes[1].plot(freqs[mask], ph[mask] / (2*np.pi),
                         color=col, lw=1.5, label=name)
        except Exception as exc:
            print(f"  {name}: {exc}")

    axes[0].set_xlabel("Frequency [Hz]")
    axes[0].set_ylabel(r"$|\tilde{h}(f)|$")
    axes[0].set_title("IMRPhenom Family — Amplitude Spectra (spin1z=0.3)")
    axes[0].legend(fontsize=9)
    axes[0].set_xlim(20, 2048)

    axes[1].set_xlabel("Frequency [Hz]")
    axes[1].set_ylabel(r"Phase / $2\pi$")
    axes[1].set_title("IMRPhenom Family — Accumulated Phase")
    axes[1].legend(fontsize=9)
    axes[1].set_xscale("log")
    plt.tight_layout()
    plt.show()

    for name, res in results.items():
        print(f"  {name:20s}  {res['time']*1e3:7.2f} ms")
else:
    print("PyCBC not available — skipping.")


---
## 5. Numerical-Relativity Surrogates <a id="5-surrogates"></a>

### 5.1 From NR Grids to Fast Surrogates

Numerical Relativity (NR) solves the full Einstein equations on a computational grid. A single NR simulation can take **days to weeks** on a supercomputer. The surrogate approach:

1. Runs $\mathcal{O}(100\text{–}1000)$ NR simulations covering a compact parameter space.
2. Decomposes waveforms using **reduced basis** (singular value decomposition).
3. Interpolates the empirical coefficients across parameter space with Gaussian processes or neural networks.
4. At evaluation time, the surrogate reconstructs the full waveform in **milliseconds** with NR-level accuracy.

Key surrogate models:

| Surrogate | Regime | Modes | Speed |
|---|---|---|---|
| NRHybSur3dq8 | $q \leq 8$, $\chi_i \leq 0.8$, aligned | $\ell\leq4$ | ~50 ms |
| NRSur7dq4 | $q \leq 4$, precessing | $\ell\leq4$ | ~100 ms |
| BHPTNRSurrogate | Extreme mass ratio | $\ell\leq10$ | ~10 ms |

### 5.2 NRHybSur3dq8 in PyCBC

> **Note**: NR surrogates require `gwsurrogate` (`pip install gwsurrogate`) and a pre-downloaded data file. The code below shows the interface; if the package is not available it falls back to a schematic plot.


In [ ]:
# ── 5.2  NR surrogate waveform ──────────────────────────────────────────────
NR_AVAILABLE = False
try:
    import gwsurrogate
    NR_AVAILABLE = True
    print(f"gwsurrogate {gwsurrogate.__version__} available ✓")
except ImportError:
    print("gwsurrogate not found. Install with: pip install gwsurrogate")
    print("Then download the data: import gwsurrogate; gwsurrogate.catalog.pull('NRHybSur3dq8')")

if NR_AVAILABLE:
    # Load surrogate (will download first time, ~200 MB)
    sur = gwsurrogate.LoadSurrogate("NRHybSur3dq8")
    q_val  = 2.0          # mass ratio m1/m2
    chi1z  = 0.3
    chi2z  = -0.2
    f_ref  = 20.0         # Hz
    M_tot  = 65.0         # M_sun
    dist_Mpc = 410.0

    t_sur, h_modes, dyn = sur(
        q_val, [0,0,chi1z], [0,0,chi2z],
        dt=1/4096, f_low=f_ref, M=M_tot, dist_mpc=dist_Mpc,
        mode_list=[(2,2),(2,-2),(3,3),(4,4)]
    )
    h22 = h_modes[(2,2)]
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(t_sur, h22.real, lw=1.0, label=r"$h_{22}$ (NRHybSur3dq8)", color="darkgreen")
    ax.set_xlabel("Time [s] (merger at $t=0$)")
    ax.set_ylabel(r"$\mathrm{Re}\,h_{22}$")
    ax.set_title(f"NRHybSur3dq8 — $q={q_val}$, $\\chi_{{1z}}={chi1z}$, $\\chi_{{2z}}={chi2z}$")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Surrogate duration: {t_sur[-1]-t_sur[0]:.2f} s, {len(t_sur)} samples")

else:
    # Schematic demonstration using a sine-Gaussian chirp as a proxy
    print("Showing a schematic NR-surrogate-like waveform (proxy)")
    t = np.linspace(-2.0, 0.05, 8192)
    # Chirp amplitude + oscillation
    tau = np.abs(t + 0.001)
    amp_nr = np.where(t < 0, tau**(-1/4), np.exp(-t/0.02))
    omega  = 2*np.pi * 100 * (0.001 / (tau + 0.001))**(3/8)
    phase  = np.cumsum(omega) / 8192
    h_nr   = amp_nr * np.cos(phase)
    h_nr  /= np.max(np.abs(h_nr))

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(t, h_nr, lw=0.8, color="darkgreen", label="NR surrogate (schematic proxy)")
    ax.axvline(0, color="k", ls="--", lw=0.8, label="Merger")
    ax.set_xlabel("Time [s] (merger at $t=0$)")
    ax.set_ylabel(r"$h_+$ (normalised)")
    ax.set_title("Schematic NR-Surrogate-Like Waveform (install gwsurrogate for real)")
    ax.legend()
    plt.tight_layout()
    plt.show()


---
## 6. Physical Effects on Waveforms <a id="6-physics"></a>

### 6.1 Spin Precession

When the binary spins $\boldsymbol{S}_{1,2}$ are misaligned with the orbital angular momentum $\boldsymbol{L}$, the orbital plane **precesses** around the total angular momentum $\boldsymbol{J} = \boldsymbol{L} + \boldsymbol{S}_1 + \boldsymbol{S}_2$ on the *precession timescale* $t_{\rm prec} \sim t_{\rm orb}/v^2$.

The dominant effect on the observed strain is an **amplitude and phase modulation** at the precession frequency. In the time domain, this produces a characteristic "double-helix" envelope.

Key quantity: the **effective precessing spin**

$$\chi_p \equiv \frac{1}{A_1 m_1^2}\max\!\left(A_1 S_{1\perp},\, A_2 S_{2\perp}\right)$$

with $A_1 = 2 + 3m_2/(2m_1)$, $A_2 = 2 + 3m_1/(2m_2)$.


In [ ]:
# ── 6.1  Spin precession: aligned vs precessing waveform ───────────────────
if PYCBC_AVAILABLE:
    common = dict(
        mass1=m1, mass2=m2,
        delta_f=delta_f, f_lower=f_lower,
        distance=410.0, inclination=0.4,
    )

    configs = [
        ("IMRPhenomD",   "Aligned spin\n($\\chi_{1z}=0.6$)",
         dict(spin1z=0.6, spin2z=0.0)),
        ("IMRPhenomPv2", "Precessing\n($\\chi_{1z}=0.6,\\ \\chi_{1x}=0.3$)",
         dict(spin1z=0.6, spin2z=0.0,
              spin1x=0.3, spin1y=0.0,
              spin2x=0.0, spin2y=0.0)),
    ]
    colors_prec = ["royalblue", "crimson"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for (approx, label, kwargs), col in zip(configs, colors_prec):
        try:
            hp, hc = get_fd_waveform(approximant=approx, **common, **kwargs)
            freqs  = hp.sample_frequencies.numpy()
            mask   = freqs > f_lower
            axes[0].loglog(freqs[mask], np.abs(hp.numpy())[mask],
                           color=col, lw=1.8, label=label.replace("\n"," "))
        except Exception as e:
            print(f"{approx}: {e}")

    axes[0].set_xlabel("Frequency [Hz]")
    axes[0].set_ylabel(r"$|\tilde h_+(f)|$")
    axes[0].set_title("Aligned vs Precessing — Amplitude")
    axes[0].legend(fontsize=9)
    axes[0].set_xlim(20, 2048)

    # Time-domain comparison via IFFT
    for (approx, label, kwargs), col in zip(configs, colors_prec):
        try:
            hp, _ = get_fd_waveform(approximant=approx, **common, **kwargs)
            hp_td = hp.to_timeseries()
            t_td  = hp_td.sample_times.numpy()
            t_merge = t_td[-1]
            mask_t = t_td > (t_merge - 1.0)
            axes[1].plot(t_td[mask_t] - t_merge,
                         hp_td.numpy()[mask_t],
                         color=col, lw=1.0,
                         label=label.replace("\n"," "))
        except Exception as e:
            print(f"{approx} time-domain: {e}")

    axes[1].set_xlabel("Time before merger [s]")
    axes[1].set_ylabel(r"$h_+(t)$")
    axes[1].set_title("Last 1 s: Aligned vs Precessing")
    axes[1].legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("PyCBC not available — skipping.")


### 6.2 Orbital Eccentricity

Most isolated binary mergers are expected to be essentially **circular** by the time they enter the LIGO band ($e \lesssim 10^{-4}$ for field binaries). However, dynamically-formed binaries in globular clusters or galactic nuclei can have measurable eccentricity at detection ($e \sim 0.1$).

Eccentricity modifies the waveform at each PN order through the **Peters-Mathews** enhancement factors:

$$\dot{E}_{\rm GW} \propto f(e) \equiv \frac{1 + \tfrac{73}{24}e^2 + \tfrac{37}{96}e^4}{(1-e^2)^{7/2}}$$

The time-domain signature is an **oscillating SNR** ("pericenter bursts") that is absent in circular waveforms.


In [ ]:
# ── 6.2  Effect of eccentricity on waveform amplitude ──────────────────────
# We illustrate the Peters-Mathews f(e) enhancement and schematic waveforms

e_vals = np.linspace(0, 0.9, 500)

def peters_f(e):
    return (1 + 73/24*e**2 + 37/96*e**4) / (1 - e**2)**(7/2)

def peters_g(e):
    # Lum. enhancement for circular orbit reference
    return (1 + 121/304*e**2) * (1 - e**2)**(5/2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].semilogy(e_vals, peters_f(e_vals), "darkorange", lw=2,
                 label=r"$f(e)$ — GW power enhancement")
axes[0].semilogy(e_vals, 1/peters_g(e_vals), "purple", lw=2, ls="--",
                 label=r"$1/g(e)$ — merger time shortening")
axes[0].set_xlabel("Eccentricity $e$")
axes[0].set_ylabel("Enhancement factor")
axes[0].set_title("Peters-Mathews Enhancement Factors")
axes[0].legend()
axes[0].set_xlim(0, 0.9)

# Schematic: eccentric waveform vs circular
t = np.linspace(0, 2.0, 8000)
omega_circ = 2*np.pi*50 * (1 - t/2.5)**(-3/8)
phase_circ = np.cumsum(omega_circ) / 8000 * 2.0
h_circ = (1 - t/2.5)**(1/4) * np.cos(phase_circ)

# Eccentric: modulated amplitude at twice orbital freq
e_demo = 0.3
t_orb  = 1.0 / 50.0
mod    = 1 + e_demo * np.cos(2*np.pi*t/t_orb)   # simple modulation proxy
h_ecc  = mod * (1 - t/2.5)**(1/4) * np.cos(phase_circ)

axes[1].plot(t[:3000], h_circ[:3000], "royalblue", lw=1.0, alpha=0.8, label="Circular ($e=0$)")
axes[1].plot(t[:3000], h_ecc[:3000],  "crimson",   lw=1.0, alpha=0.8, label=f"Eccentric ($e={e_demo}$, schematic)")
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel(r"$h_+(t)$ (normalised)")
axes[1].set_title("Circular vs Eccentric Waveform (schematic)")
axes[1].legend()
plt.tight_layout()
plt.show()
print("Eccentricity e=0.3 enhances GW power by factor", f"{peters_f(0.3):.2f}")
print("Eccentricity e=0.7 enhances GW power by factor", f"{peters_f(0.7):.1f}")


### 6.3 Tidal Deformability $\Lambda$ (Binary Neutron Stars)

For BNS, neutron stars are not point masses — they are deformed by the companion's tidal field. This is characterised by the **dimensionless tidal deformability**:

$$\Lambda = \frac{2}{3} k_2 \left(\frac{R}{GM/c^2}\right)^5$$

where $k_2$ is the quadrupolar Love number and $R$ the NS radius.

Tidal effects enter the GW phase at **5PN order** and beyond:

$$\Psi_{\rm tidal}(f) = -\frac{3}{128\eta v^5}\left[\frac{39}{2}\tilde\Lambda\right]v^{10} + \ldots$$

with the combined tidal parameter

$$\tilde\Lambda = \frac{16}{13}\frac{(m_1 + 12m_2)m_1^4 \Lambda_1 + (m_2 + 12m_1)m_2^4 \Lambda_2}{(m_1+m_2)^5}$$

The measurement of $\tilde\Lambda = 300^{+420}_{-230}$ from GW170817 constrained the NS equation of state and favoured a relatively soft EOS.


In [ ]:
# ── 6.3  Tidal effects on BNS waveform phase ───────────────────────────────
if PYCBC_AVAILABLE:
    # BNS parameters — GW170817-like
    m_ns = 1.4   # M_sun
    configs_bns = [
        ("IMRPhenomD_NRTidalv2", "No tides ($\\Lambda=0$)",   0,   0),
        ("IMRPhenomD_NRTidalv2", "$\\Lambda=300$",           300, 300),
        ("IMRPhenomD_NRTidalv2", "$\\Lambda=800$",           800, 800),
    ]
    colors_tidal = ["royalblue", "darkorange", "crimson"]
    freqs_ref = None
    phases_tidal = {}

    fig, axes = plt.subplots(2, 1, figsize=(13, 7))

    for (approx, label, L1, L2), col in zip(configs_bns, colors_tidal):
        try:
            hp, _ = get_fd_waveform(
                approximant=approx,
                mass1=m_ns, mass2=m_ns,
                lambda1=L1, lambda2=L2,
                spin1z=0.0, spin2z=0.0,
                delta_f=delta_f/4,
                f_lower=20.0,
                distance=40.0,    # GW170817 at ~40 Mpc
            )
            freqs = hp.sample_frequencies.numpy()
            mask  = (freqs > 20) & (freqs < 2000)
            axes[0].loglog(freqs[mask], np.abs(hp.numpy())[mask],
                           color=col, lw=1.8, label=label)
            phases_tidal[label] = (freqs, np.unwrap(np.angle(hp.numpy())))
        except Exception as exc:
            print(f"  {approx} ({label}): {exc}")

    axes[0].set_xlabel("Frequency [Hz]")
    axes[0].set_ylabel(r"$|\tilde h(f)|$")
    axes[0].set_title("BNS Waveforms — Tidal Deformability Effect on Amplitude")
    axes[0].legend()

    # Phase difference relative to Λ=0
    ref_label = "No tides ($\\Lambda=0$)"
    if ref_label in phases_tidal:
        f_ref, ph_ref = phases_tidal[ref_label]
        for (approx, label, L1, L2), col in zip(configs_bns[1:], colors_tidal[1:]):
            if label in phases_tidal:
                f_l, ph_l = phases_tidal[label]
                n = min(len(f_ref), len(f_l))
                mask = (f_ref[:n] > 20) & (f_ref[:n] < 2000)
                axes[1].plot(f_ref[:n][mask],
                             (ph_l[:n][mask] - ph_ref[:n][mask]) / (2*np.pi),
                             color=col, lw=1.8, label=f"$\\Delta\\Psi$: {label} vs no tides")
    axes[1].set_xlabel("Frequency [Hz]")
    axes[1].set_ylabel(r"Phase difference $\Delta\Psi / 2\pi$ [cycles]")
    axes[1].set_title("Tidal Phase Accumulation (negative = faster merger)")
    axes[1].set_xscale("log")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

else:
    # Analytic illustration of tidal phase
    f_vals = np.linspace(20, 2000, 2000)
    m_ns   = 1.4 * 4.926e-6   # seconds
    M_tot  = 2 * m_ns
    eta_bns = 0.25

    def tidal_phase(f, Lambda_tilde):
        v = (np.pi * M_tot * f)**(1/3)
        return -(3/(128*eta_bns*v**5)) * (39/2 * Lambda_tilde) * v**10

    fig, ax = plt.subplots(figsize=(11, 5))
    for Lam, col, lbl in [(300,"darkorange","$\\tilde\\Lambda=300$"),
                           (800,"crimson","$\\tilde\\Lambda=800$")]:
        dph = (tidal_phase(f_vals, Lam) - tidal_phase(f_vals, 0)) / (2*np.pi)
        ax.semilogx(f_vals, dph, color=col, lw=2, label=lbl)
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"$\Delta\Psi / 2\pi$ [cycles]")
    ax.set_title("Tidal Phase Shift (analytic 5PN, BNS $m_1=m_2=1.4\\,M_\\odot$)")
    ax.legend()
    plt.tight_layout()
    plt.show()


---
## 7. Comparing Waveform Families: Mismatch & Cost <a id="7-comparison"></a>

### 7.1 The Mismatch Metric

The **overlap** between two normalised waveforms $\hat h_1$ and $\hat h_2$ is

$$\mathcal{O}(h_1, h_2) = \max_{t_c, \phi_c} \frac{\langle h_1 | h_2 \rangle}{\sqrt{\langle h_1|h_1\rangle \langle h_2|h_2\rangle}}$$

The **mismatch** is $\mathcal{M}_{\rm mm} = 1 - \mathcal{O}$. A mismatch of $\mathcal{M}_{\rm mm}$ causes a fractional loss in **detection volume** of approximately

$$\frac{\Delta V}{V} \approx 3\,\mathcal{M}_{\rm mm}$$

so $\mathcal{M}_{\rm mm} = 3\%$ means ~9% volume loss. Template banks are designed with $\mathcal{M}_{\rm mm} < 3\%$.

### 7.2 Systematic Mismatch Survey

We compute the match between TaylorF2, SEOBNRv4_ROM, and IMRPhenomD over a grid of total masses.


In [ ]:
# ── 7.2  Mismatch survey over total mass ────────────────────────────────────
if PYCBC_AVAILABLE:
    from pycbc.filter import match
    from pycbc.psd import aLIGOZeroDetHighPower

    q_ratio = 2.0          # mass ratio
    masses  = np.array([20, 30, 40, 60, 80, 100, 150])  # total mass M_sun

    approx_pairs = [
        ("TaylorF2",   "IMRPhenomD"),
        ("TaylorF2",   "SEOBNRv4_ROM"),
        ("IMRPhenomD", "SEOBNRv4_ROM"),
    ]
    mismatch_results = {pair: [] for pair in approx_pairs}

    for M_tot in masses:
        m1_grid = M_tot * q_ratio / (1 + q_ratio)
        m2_grid = M_tot / (1 + q_ratio)

        # Generate PSD at the right length
        ref_df   = 0.125
        ref_flen = int(4096 / ref_df) // 2 + 1
        psd = aLIGOZeroDetHighPower(ref_flen, ref_df, 20.0)

        for (a1, a2) in approx_pairs:
            try:
                h1, _ = get_fd_waveform(approximant=a1,
                                        mass1=m1_grid, mass2=m2_grid,
                                        spin1z=0.0, spin2z=0.0,
                                        delta_f=ref_df, f_lower=20.0, distance=1.0)
                h2, _ = get_fd_waveform(approximant=a2,
                                        mass1=m1_grid, mass2=m2_grid,
                                        spin1z=0.0, spin2z=0.0,
                                        delta_f=ref_df, f_lower=20.0, distance=1.0)
                # Resize to PSD length
                h1.resize(ref_flen)
                h2.resize(ref_flen)
                m, _ = match(h1, h2, psd=psd, low_frequency_cutoff=20.0)
                mismatch_results[(a1,a2)].append(1 - m)
            except Exception as exc:
                mismatch_results[(a1,a2)].append(np.nan)
                print(f"  {a1} vs {a2} at M={M_tot}: {exc}")

    fig, ax = plt.subplots(figsize=(10, 5))
    colors_mm = ["royalblue", "crimson", "seagreen"]
    for (pair, col) in zip(approx_pairs, colors_mm):
        mm_arr = np.array(mismatch_results[pair])
        ax.semilogy(masses, mm_arr * 100, marker="o", color=col, lw=2,
                    label=f"{pair[0]} vs {pair[1]}")
    ax.axhline(3, color="k", ls="--", lw=1.5, label="3% threshold")
    ax.set_xlabel(r"Total mass $M_{\rm tot}$ [$M_\odot$]")
    ax.set_ylabel("Mismatch [%]")
    ax.set_title("Mismatch between Waveform Families ($q=2$, no spin)")
    ax.legend(fontsize=9)
    ax.set_ylim(1e-3, 20)
    plt.tight_layout()
    plt.show()

    print("\nMismatch table (%):")
    header = f"{'M_tot':>8}" + "".join(f"  {a1[:7]}v{a2[:7]}":>22 for a1,a2 in approx_pairs)
    print(header)
    for i, M in enumerate(masses):
        row = f"{M:>8.0f}" + "".join(
            f"  {mismatch_results[pair][i]*100:>18.4f}%" for pair in approx_pairs
        )
        print(row)

else:
    print("PyCBC not available — showing illustrative mismatch plot.")
    masses_demo = np.array([20, 30, 40, 60, 80, 100, 150], dtype=float)
    mm_tf2_phenom = np.array([0.02, 0.08, 0.5, 2.1, 5.2, 8.0, 12.0])
    mm_tf2_seob   = np.array([0.01, 0.05, 0.3, 1.5, 4.0, 7.0, 11.0])
    mm_phenom_seob= np.array([0.01, 0.02, 0.05, 0.2, 0.5, 1.0, 2.0])

    fig, ax = plt.subplots(figsize=(10,5))
    ax.semilogy(masses_demo, mm_tf2_phenom,  "royalblue", marker="o", lw=2, label="TaylorF2 vs IMRPhenomD")
    ax.semilogy(masses_demo, mm_tf2_seob,    "crimson",   marker="s", lw=2, label="TaylorF2 vs SEOBNRv4_ROM")
    ax.semilogy(masses_demo, mm_phenom_seob, "seagreen",  marker="^", lw=2, label="IMRPhenomD vs SEOBNRv4_ROM")
    ax.axhline(3, color="k", ls="--", lw=1.5, label="3% threshold")
    ax.set_xlabel(r"Total mass $M_{\rm tot}$ [$M_\odot$]")
    ax.set_ylabel("Mismatch [%]  (illustrative)")
    ax.set_title("Mismatch between Waveform Families (illustrative)")
    ax.legend(fontsize=9)
    ax.set_ylim(1e-3, 20)
    plt.tight_layout()
    plt.show()


### 7.3 Computational Cost Benchmarks

Beyond accuracy, waveform generation speed determines whether an approximant can be used in:

* **Template banks** (millions of waveforms needed)
* **Bayesian parameter estimation** (MCMC/nested sampling: $10^5$–$10^6$ evaluations)
* **Online searches** (real-time SNR computation)

The table below gives typical generation times for a $60\,M_\odot$ BBH at 4096 Hz sampling:

| Approximant | Type | ~Cost (ms) | Notes |
|---|---|---|---|
| TaylorF2 | FD PN | 0.1 | Closed-form; cheapest |
| IMRPhenomD | FD phenom | 0.5 | Near-analytic |
| IMRPhenomPv2 | FD phenom+prec | 2 | Double-frame precession |
| SEOBNRv4_ROM | FD EOB ROM | 5–50 | ROM speed-up over full EOB |
| SEOBNRv4 (full) | TD EOB ODE | 500–5000 | Full ODE integration |
| NRHybSur3dq8 | TD NR surr | 50–200 | SVD + GP interpolation |
| NRSur7dq4 | TD NR surr | 100–500 | Full precessing |


In [ ]:
# ── 7.3  Timing benchmark ────────────────────────────────────────────────────
if PYCBC_AVAILABLE:
    bench_approx = [
        ("TaylorF2",       "fd", dict(spin1z=0.0,  spin2z=0.0)),
        ("IMRPhenomD",     "fd", dict(spin1z=0.3,  spin2z=-0.1)),
        ("IMRPhenomPv2",   "fd", dict(spin1z=0.3,  spin2z=-0.1,
                                      spin1x=0.2, spin1y=0.0,
                                      spin2x=0.0, spin2y=0.0,
                                      inclination=0.4)),
        ("SEOBNRv4_ROM",   "fd", dict(spin1z=0.3,  spin2z=-0.1)),
    ]
    common_bench = dict(mass1=40.0, mass2=20.0, delta_f=0.125,
                        f_lower=20.0, distance=410.0)

    NREP = 5
    timing = {}
    for name, domain, kwargs in bench_approx:
        times = []
        for _ in range(NREP):
            t0 = time.perf_counter()
            try:
                get_fd_waveform(approximant=name, **common_bench, **kwargs)
                times.append(time.perf_counter() - t0)
            except Exception as exc:
                print(f"  {name}: {exc}")
                break
        if times:
            timing[name] = np.median(times) * 1e3  # ms

    names  = list(timing.keys())
    t_vals = [timing[n] for n in names]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(names, t_vals,
                   color=["royalblue","darkorange","seagreen","crimson"])
    ax.set_xscale("log")
    ax.set_xlabel("Median generation time [ms]")
    ax.set_title(r"Waveform Generation Cost ($M_1=40, M_2=20\,M_\odot$, $f_{\rm low}=20$ Hz)")
    for bar, val in zip(bars, t_vals):
        ax.text(val * 1.05, bar.get_y() + bar.get_height()/2,
                f"{val:.2f} ms", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

    print("\nTiming results (median over 5 runs):")
    for n, t in timing.items():
        print(f"  {n:25s}  {t:8.3f} ms")
else:
    print("PyCBC not available — showing reference timing chart.")
    names_ref = ["TaylorF2", "IMRPhenomD", "IMRPhenomPv2", "SEOBNRv4_ROM", "SEOBNRv4 (full ODE)"]
    t_ref     = [0.1, 0.5, 2.0, 20.0, 2000.0]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(names_ref, t_ref, color=["royalblue","darkorange","seagreen","crimson","purple"])
    ax.set_xscale("log")
    ax.set_xlabel("Approximate generation time [ms]  (reference values)")
    ax.set_title("Waveform Computational Cost (reference)")
    plt.tight_layout()
    plt.show()


---
## 8. Signal Injection & Recovery Analysis <a id="8-injection"></a>

Signal injection tests are the gold standard for validating a waveform model and detection pipeline. The workflow is:

1. **Generate an injected signal** $h_{\rm inj}(t)$ with known parameters using a "true" waveform model.
2. **Add to real (or simulated) noise** $n(t)$: $s(t) = n(t) + h_{\rm inj}(t)$.
3. **Recover the signal** by matched-filtering $s(t)$ against templates from a *different* (or same) approximant.
4. **Measure the fitting factor** (maximum match over template-bank parameters) and recovered SNR.

The **fitting factor** is

$$\mathcal{F} = \max_{\boldsymbol\lambda_{\rm template}} \frac{\langle s | h_{\rm template}(\boldsymbol\lambda)\rangle}{\|s\|\,\|h_{\rm template}\|}$$

If $\mathcal{F} < 0.97$ (i.e. mismatch > 3%), the template bank "misses" real events.

### 8.1 Simulated Noise & Injection


In [ ]:
# ── 8.1  Gaussian noise + signal injection ──────────────────────────────────
if PYCBC_AVAILABLE:
    from pycbc.noise import noise_from_psd
    from pycbc.filter import matched_filter

    # PSD
    sample_rate = 4096
    T_noise     = 64     # seconds of noise
    delta_t_n   = 1.0 / sample_rate
    delta_f_n   = 1.0 / T_noise
    flen        = T_noise * sample_rate // 2 + 1

    psd_inj = aLIGOZeroDetHighPower(flen, delta_f_n, 20.0)

    # Generate coloured Gaussian noise
    np.random.seed(42)
    strain_noise = noise_from_psd(T_noise * sample_rate, delta_t_n, psd_inj,
                                  low_frequency_cutoff=20.0)

    # Generate injection: IMRPhenomD signal (the "true" signal)
    m1_inj, m2_inj = 36.0, 29.0
    inj_params = dict(
        approximant="IMRPhenomD",
        mass1=m1_inj, mass2=m2_inj,
        spin1z=0.3, spin2z=-0.1,
        delta_t=delta_t_n,
        f_lower=20.0,
        distance=500.0,
    )
    hp_inj, hc_inj = get_td_waveform(**inj_params)
    hp_inj.resize(len(strain_noise))

    # Inject: place signal 10 s before end
    t_inj_gps = strain_noise.start_time + T_noise - 10.0
    hp_inj_shifted = hp_inj.cyclic_time_shift(
        t_inj_gps - hp_inj.start_time - float(hp_inj.duration)
    )
    strain_signal = strain_noise.copy()
    strain_signal.data[:] += hp_inj_shifted.data[:]

    # Compute optimal SNR for reference
    hp_fd_inj, _ = get_fd_waveform(
        approximant="IMRPhenomD",
        mass1=m1_inj, mass2=m2_inj,
        spin1z=0.3, spin2z=-0.1,
        delta_f=delta_f_n, f_lower=20.0, distance=500.0
    )
    hp_fd_inj.resize(flen)
    opt_snr = sigma(hp_fd_inj, psd=psd_inj, low_frequency_cutoff=20.0)
    print(f"Optimal SNR of injected signal: {opt_snr:.2f}")

    # Plot 2 s around injection
    t_arr   = strain_noise.sample_times.numpy()
    t_inj_c = float(t_inj_gps)
    window  = (t_arr > t_inj_c - 1.5) & (t_arr < t_inj_c + 0.5)

    fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
    axes[0].plot(t_arr[window] - t_inj_c, strain_noise.numpy()[window],
                 "gray", lw=0.5, alpha=0.8, label="Noise only")
    axes[0].set_ylabel("Strain")
    axes[0].set_title("Detector Noise")
    axes[0].legend()

    axes[1].plot(t_arr[window] - t_inj_c, hp_inj_shifted.numpy()[window],
                 "darkorange", lw=1.2, label="Injected signal")
    axes[1].set_ylabel("Strain")
    axes[1].set_title(f"Injected IMRPhenomD Signal (opt. SNR = {opt_snr:.1f})")
    axes[1].legend()

    axes[2].plot(t_arr[window] - t_inj_c, strain_signal.numpy()[window],
                 "royalblue", lw=0.6, alpha=0.9, label="Noise + Signal")
    axes[2].set_xlabel("Time relative to injection [s]")
    axes[2].set_ylabel("Strain")
    axes[2].set_title("Noise + Injected Signal")
    axes[2].legend()

    plt.tight_layout()
    plt.show()
else:
    print("PyCBC not available — skipping injection demo.")


### 8.2 Matched-Filter Recovery with Cross-Approximant Templates

We now apply matched filtering to the noisy data using:
- **Same** approximant (IMRPhenomD) — "perfect" template
- **Cross** approximant (TaylorF2) — measuring the impact of waveform systematics

This directly quantifies how much SNR is lost when the template family does not match the signal.


In [ ]:
# ── 8.2  Cross-approximant matched-filter recovery ──────────────────────────
if PYCBC_AVAILABLE:
    from pycbc.filter import matched_filter, resample_to_delta_t
    from pycbc.types import TimeSeries as PyCBCTS

    recovery_approx = [
        ("IMRPhenomD (same)",   dict(approximant="IMRPhenomD",
                                     spin1z=0.3, spin2z=-0.1)),
        ("TaylorF2 (cross)",    dict(approximant="TaylorF2",
                                     spin1z=0.0, spin2z=0.0)),
        ("IMRPhenomPv2 (cross)",dict(approximant="IMRPhenomPv2",
                                     spin1z=0.3, spin2z=-0.1,
                                     spin1x=0.0, spin1y=0.0,
                                     spin2x=0.0, spin2y=0.0,
                                     inclination=0.0)),
    ]
    cols_rec = ["seagreen", "royalblue", "crimson"]

    fig, axes = plt.subplots(len(recovery_approx), 1,
                             figsize=(13, 4*len(recovery_approx)), sharex=True)
    snr_peaks = {}

    for ax, (label, kwargs), col in zip(axes, recovery_approx, cols_rec):
        try:
            ht, _ = get_fd_waveform(
                mass1=m1_inj, mass2=m2_inj,
                delta_f=delta_f_n, f_lower=20.0, distance=1.0,
                **kwargs
            )
            ht.resize(flen)
            snr_ts = matched_filter(ht, strain_signal.to_frequencyseries(),
                                    psd=psd_inj,
                                    low_frequency_cutoff=20.0)
            snr_arr = np.abs(snr_ts.numpy())
            t_snr   = snr_ts.sample_times.numpy()
            peak    = np.max(snr_arr)
            snr_peaks[label] = peak

            ax.plot(t_snr - t_inj_c, snr_arr,
                    color=col, lw=0.8, alpha=0.8, label=f"{label}  (peak SNR={peak:.1f})")
            ax.axvline(0, color="k", ls="--", lw=1, label="Injection time")
            ax.set_ylabel("SNR $\\rho(t)$")
            ax.set_title(f"Matched Filter SNR — {label}")
            ax.legend(fontsize=9)
            ax.set_xlim(-2, 1)
        except Exception as exc:
            print(f"  {label}: {exc}")
            ax.set_visible(False)

    axes[-1].set_xlabel("Time relative to injection [s]")
    plt.tight_layout()
    plt.show()

    print("\nSNR recovery summary:")
    ref_snr = snr_peaks.get("IMRPhenomD (same)", np.nan)
    for label, pk in snr_peaks.items():
        ratio = pk / ref_snr if ref_snr > 0 else np.nan
        print(f"  {label:35s}  peak SNR = {pk:.2f}  ({ratio*100:.1f}% of optimal)")
else:
    print("PyCBC not available — skipping matched-filter recovery.")


### 8.3 Efficiency Curves and Fitting Factor

The **fitting factor** (FF) as a function of signal parameters tells us which part of parameter space a given template bank covers efficiently.

Here we sweep the injected chirp mass $\mathcal{M}_c$ and plot the fraction of the optimal SNR recovered by a TaylorF2 template at the *same* parameters — demonstrating the degradation at high mass where PN breaks down.


In [ ]:
# ── 8.3  Fitting factor vs total mass ───────────────────────────────────────
if PYCBC_AVAILABLE:
    from pycbc.filter import match as pycbc_match

    M_tot_vals = np.array([20, 30, 40, 60, 80, 100])
    q_ff       = 2.0

    ff_phenom_vs_tf2  = []
    ff_seob_vs_tf2    = []
    ff_phenom_vs_seob = []

    for M_t in M_tot_vals:
        m1f = M_t * q_ff / (1 + q_ff)
        m2f = M_t       / (1 + q_ff)

        df_ff   = 0.25
        flen_ff = int(2048 / df_ff) // 2 + 1
        psd_ff  = aLIGOZeroDetHighPower(flen_ff, df_ff, 20.0)

        try:
            h_phenom, _ = get_fd_waveform(
                approximant="IMRPhenomD",
                mass1=m1f, mass2=m2f, spin1z=0.0, spin2z=0.0,
                delta_f=df_ff, f_lower=20.0, distance=1.0)
            h_tf2, _ = get_fd_waveform(
                approximant="TaylorF2",
                mass1=m1f, mass2=m2f, spin1z=0.0, spin2z=0.0,
                delta_f=df_ff, f_lower=20.0, distance=1.0)
            h_seob, _ = get_fd_waveform(
                approximant="SEOBNRv4_ROM",
                mass1=m1f, mass2=m2f, spin1z=0.0, spin2z=0.0,
                delta_f=df_ff, f_lower=20.0, distance=1.0)

            for h in [h_phenom, h_tf2, h_seob]:
                h.resize(flen_ff)

            m1_val, _ = pycbc_match(h_phenom, h_tf2,  psd=psd_ff, low_frequency_cutoff=20.0)
            m2_val, _ = pycbc_match(h_seob,   h_tf2,  psd=psd_ff, low_frequency_cutoff=20.0)
            m3_val, _ = pycbc_match(h_phenom, h_seob, psd=psd_ff, low_frequency_cutoff=20.0)
        except Exception as exc:
            m1_val = m2_val = m3_val = np.nan
            print(f"  M={M_t}: {exc}")

        ff_phenom_vs_tf2.append(m1_val)
        ff_seob_vs_tf2.append(m2_val)
        ff_phenom_vs_seob.append(m3_val)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(M_tot_vals, np.array(ff_phenom_vs_tf2)*100,  "royalblue",
            marker="o", lw=2, label="IMRPhenomD ↔ TaylorF2")
    ax.plot(M_tot_vals, np.array(ff_seob_vs_tf2)*100,    "crimson",
            marker="s", lw=2, label="SEOBNRv4_ROM ↔ TaylorF2")
    ax.plot(M_tot_vals, np.array(ff_phenom_vs_seob)*100, "seagreen",
            marker="^", lw=2, label="IMRPhenomD ↔ SEOBNRv4_ROM")
    ax.axhline(97, color="k", ls="--", lw=1.5, label="97% (3% mismatch threshold)")
    ax.set_xlabel(r"Total mass $M_{\rm tot}$ [$M_\odot$]")
    ax.set_ylabel("Fitting Factor [%]")
    ax.set_title("Fitting Factor between Waveform Families ($q=2$, non-spinning)")
    ax.set_ylim(70, 100.5)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print("\nFitting factor table (%):")
    print(f"{'M_tot':>6}  {'Phenom vs TF2':>16}  {'SEOB vs TF2':>14}  {'Phenom vs SEOB':>16}")
    for M_t, f1, f2, f3 in zip(M_tot_vals, ff_phenom_vs_tf2, ff_seob_vs_tf2, ff_phenom_vs_seob):
        print(f"{M_t:>6.0f}  {f1*100:>16.2f}  {f2*100:>14.2f}  {f3*100:>16.2f}")

else:
    # Illustrative plot
    M_demo  = np.array([20, 30, 40, 60, 80, 100], dtype=float)
    ff_demo = np.array([99.8, 99.2, 97.5, 92.0, 85.0, 78.0])
    fig, ax = plt.subplots(figsize=(10,5))
    ax.plot(M_demo, ff_demo, "royalblue", marker="o", lw=2,
            label="IMRPhenomD vs TaylorF2 (illustrative)")
    ax.axhline(97, color="k", ls="--", lw=1.5, label="97% threshold")
    ax.set_xlabel(r"$M_{\rm tot}$ [$M_\odot$]")
    ax.set_ylabel("Fitting Factor [%]  (illustrative)")
    ax.set_title("Fitting Factor — IMRPhenomD vs TaylorF2 (illustrative)")
    ax.set_ylim(70, 100.5)
    ax.legend()
    plt.tight_layout()
    plt.show()


---
## 9. Additional Topics <a id="9-additional"></a>

### 9.1 Waveform Models for LISA and Einstein Telescope

Next-generation detectors require extended waveform models:

| Detector | $f$ range | New requirements |
|---|---|---|
| **LISA** | 0.1 mHz – 0.1 Hz | SMBHB, EMRI, milliHz GW; years-long signals |
| **Einstein Telescope** | 2 Hz – 10 kHz | Low-mass BNS at cosmological distances; 3PN+ tides |
| **Cosmic Explorer** | 5 Hz – 30 kHz | Sub-solar-mass binaries; accurate tidal waveforms |

For LISA massive black-hole binaries, models must include:
- **EMRI** waveforms (extreme mass ratio inspiral) using black-hole perturbation theory
- Doppler phase shifts from LISA's orbital motion
- Strong-field spin precession ($v/c \sim 0.5$) — beyond PN validity

For ET/CE BNS:
- **Post-merger** waveform (2–4 kHz oscillation from the hypermassive NS remnant)
- **Multiple tidal modes** ($\Lambda_2, \Lambda_3, ...$)

### 9.2 Machine-Learning Surrogate Models

Recent years have seen a wave of ML-based approaches:

**Principal component regression** (Blackman+2017):
- Decompose NR waveforms into principal components
- Fit amplitudes with Gaussian process regression

**Reduced-order quadrature** (Chua & Vallisneri 2020):
- Build a sparse frequency grid where the waveform is evaluated
- Likelihood computation 10–100× faster

**Deep learning (RNN/LSTM/Transformer)**:
- Learn the mapping $(\boldsymbol\lambda) \to \tilde{h}(f)$ directly
- Inference in ~1 ms; accuracy challenges remain for precessing signals

**Diffusion models** (2024):
- Generative models conditioned on physical parameters
- Produce time-domain waveforms with NR fidelity


In [ ]:
# ── 9.2  PSD and horizon comparison for different detectors ─────────────────
if PYCBC_AVAILABLE:
    from pycbc.psd import (aLIGOZeroDetHighPower,
                            AdvVirgo,
                            EinsteinTelescopeP1600143)

    f_arr = np.linspace(2, 4096, 40000)
    df_plot = f_arr[1] - f_arr[0]
    flen_p  = len(f_arr) + 1

    psds_plot = {
        "aLIGO (design)":        aLIGOZeroDetHighPower(flen_p, df_plot, 2.0),
        "AdV (design)":          AdvVirgo(flen_p, df_plot, 2.0),
        "ET-D":                  EinsteinTelescopeP1600143(flen_p, df_plot, 2.0),
    }

    fig, ax = plt.subplots(figsize=(12, 5))
    colors_det = ["royalblue", "darkorange", "crimson"]
    for (name, psd), col in zip(psds_plot.items(), colors_det):
        f_psd = psd.sample_frequencies.numpy()
        mask  = (f_psd > 2) & (f_psd < 4096)
        Sn    = psd.numpy()
        ax.loglog(f_psd[mask], np.sqrt(Sn[mask]), lw=2, color=col, label=name)

    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"$\sqrt{S_n(f)}$  [strain / Hz$^{1/2}$]")
    ax.set_title("Detector Noise Curves — Current and Future")
    ax.set_xlim(2, 4096)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("PyCBC not available — skipping noise curve plot.")


---
## 10. Student Exercises <a id="10-exercises"></a>

The following exercises are designed to deepen your understanding of waveform models, physical effects, and their practical implications in GW data analysis. They are ordered from foundational to advanced.

---

### Exercise 1 — PN Phase Structure

**(a)** Using the TaylorF2 phase formula, derive an expression for the **time-to-coalescence** $\tau(f)$ as a function of frequency, chirp mass $\mathcal{M}_c$, and symmetric mass ratio $\eta$. Verify numerically by computing $\tau(f)$ from the TaylorT4 ODE and comparing.

**(b)** For a $\mathcal{M}_c = 8.9\,M_\odot$ system (GW150914-like), how many GW cycles accumulate between 20 Hz and 200 Hz? Compute this analytically (leading PN order) and verify by integrating the TaylorF2 phase numerically.

**(c)** Show that a 1% error in chirp mass leads to roughly $\Delta N_{\rm cycles} \sim 3 \%\,N_{\rm cycles}$ phase shift. Discuss the implications for template bank design.

---

### Exercise 2 — Spin Precession

**(a)** For a BBH with $m_1 = 36\,M_\odot$, $m_2 = 29\,M_\odot$, $\chi_1 = 0.6$, $\theta_{LS} = 45°$ (tilt of spin 1 relative to $\hat L$), estimate the **precession frequency** $\Omega_p$ at $f_{\rm GW} = 40$ Hz using the leading-order formula:
$$\Omega_p \approx \frac{1}{2} \frac{G J}{r^3 c^2}$$

**(b)** Generate waveforms with `IMRPhenomD` (aligned) and `IMRPhenomPv2` (precessing) with $\chi_{1x} = 0.4$, $\chi_{1z} = 0.3$. Compute their mismatch as a function of inclination angle $\iota \in [0, \pi/2]$. At which inclination is precession most visible?

**(c)** Using the PyCBC `match` function, confirm that the fitting factor between the aligned and precessing waveform drops below 97% for $\iota > 30°$ (face-off orientation).

---

### Exercise 3 — Tidal Deformability and NS EOS

**(a)** Using the BNS TaylorF2 phase formula, show that $\tilde\Lambda$ is degenerate with higher PN spin terms at low frequency. At what frequency does tidal dephasing exceed 1 cycle (Δψ > 2π) for $\tilde\Lambda = 500$, $m_1=m_2=1.35\,M_\odot$?

**(b)** For the GW170817 event ($d_L \approx 40$ Mpc, $\mathcal{M}_c \approx 1.19\,M_\odot$), compute the optimal SNR for tidal-on vs tidal-off waveforms using `IMRPhenomD_NRTidalv2`. What fraction of the total SNR comes from the tidal phase above 1000 Hz?

**(c)** The Love number $k_2$ and radius $R$ depend on the nuclear EOS. Using the universal relation $\Lambda \approx 0.0094 (R/M)^7$ (Yagi & Yunes 2013), estimate $\Lambda$ for APR4 ($R \approx 11.1$ km) and SLy ($R \approx 11.4$ km) at $m = 1.4\,M_\odot$. Which EOS is more consistent with the GW170817 constraint $\tilde\Lambda < 800$?

---

### Exercise 4 — Mismatch and Detection Volume

**(a)** Prove that the fractional loss of detected event rate due to mismatch $\mathcal{M}_{\rm mm}$ is approximately $1 - (1 - \mathcal{M}_{\rm mm})^3 \approx 3\mathcal{M}_{\rm mm}$ for $\mathcal{M}_{\rm mm} \ll 1$.

**(b)** Using the PyCBC `match` function, compute the mismatch between `IMRPhenomD` and `SEOBNRv4_ROM` for a grid of total masses $M_{\rm tot} \in [20, 200]\,M_\odot$ and mass ratios $q \in [1, 8]$. Plot the mismatch as a 2D heatmap. Where in parameter space is the mismatch worst?

**(c)** For a hypothetical detector with noise curve $S_n(f) \propto f^{-4}$ (steep low-frequency noise), does the mismatch between PN and IMRPhenom get better or worse compared to aLIGO? Why?

---

### Exercise 5 — Cross-Approximant Injection-Recovery

**(a)** Inject an `SEOBNRv4_ROM` signal with $m_1=60\,M_\odot$, $m_2=20\,M_\odot$, $\chi_{1z}=0.5$, $\chi_{2z}=-0.2$ into 64 s of simulated aLIGO noise. Recover with `TaylorF2` and `IMRPhenomD` templates at the *true* masses and spins. Compute the recovered SNR for each and the ratio to the optimal SNR (fitting factor).

**(b)** Repeat Exercise 5(a) for 50 injections at random distances (uniform in co-moving volume) and compute the **detection efficiency** curve: fraction of events with recovered SNR > 8 vs. injected SNR. How does the efficiency curve change between same-approximant and cross-approximant recovery?

**(c)** Implement a simple template bank for $M_{\rm tot} \in [20, 60]\,M_\odot$, $q \in [1,4]$, non-spinning, using `TaylorF2` with minimal match $\mathcal{M} = 0.97$. Inject `IMRPhenomD` signals at 100 random points and compute the fitting factor distribution. What fraction of injections have FF < 0.97?

---

### Exercise 6 — Numerical Relativity Surrogate

**(a)** Install `gwsurrogate` and download `NRHybSur3dq8`. Generate waveforms for $q \in [1, 8]$ at fixed total mass $M_{\rm tot} = 60\,M_\odot$ and compute the match with `IMRPhenomD`. How does the match degrade at high $q$?

**(b)** Compare the $(2,2)$ mode of `NRHybSur3dq8` with `SEOBNRv4_ROM` for $q=4$, $\chi_{1z}=0.5$. Where in time/frequency do they differ most?

**(c)** Using `NRHybSur3dq8`, extract the $(3,3)$ and $(4,4)$ sub-dominant modes and compute their relative contribution to the total SNR for an equal-mass system at inclination $\iota = \pi/3$.

---

### Challenge Exercise — Full Bayesian Inference

Set up a toy Bayesian parameter estimation on a simulated signal using `bilby` (or `PyCBC Inference`):

1. Inject an `IMRPhenomPv2` signal (precessing) with known parameters.
2. Run nested sampling with `IMRPhenomD` (non-precessing) as the likelihood template.
3. Compare the posterior on $\chi_{\rm eff}$ and $\mathcal{M}_c$ with and without precession in the template.
4. Quantify the **parameter bias** introduced by using a non-precessing template.


---
## 11. References <a id="11-references"></a>

### Waveform Approximant Papers

#### Post-Newtonian
1. **Blanchet, L. (2014)** — Gravitational Radiation from Post-Newtonian Sources and Inspiralling Compact Binaries.  
   *Living Rev. Rel.* 17, 2. [arXiv:1310.1528](https://arxiv.org/abs/1310.1528)

2. **Buonanno, A., Iyer, B. R., Ochsner, E., Pan, Y. & Sathyaprakash, B. S. (2009)** — Comparison of post-Newtonian templates for compact binary inspiral signals in gravitational-wave detectors.  
   *Phys. Rev. D* 80, 084043. [arXiv:0907.0700](https://arxiv.org/abs/0907.0700)

3. **Droz, S., Knapp, D. J., Poisson, E. & Owen, B. J. (1999)** — Gravitational waves from inspiraling compact binaries: Validity of the stationary-phase approximation to the Fourier transform.  
   *Phys. Rev. D* 59, 124016. [arXiv:gr-qc/9901076](https://arxiv.org/abs/gr-qc/9901076)

#### Effective One-Body
4. **Buonanno, A. & Damour, T. (1999)** — Transition from inspiral to plunge in binary black hole coalescences.  
   *Phys. Rev. D* 59, 084006. [arXiv:gr-qc/9811091](https://arxiv.org/abs/gr-qc/9811091)

5. **Bohé, A. et al. (2017)** — Improved effective-one-body model of spinning, nonprecessing binary black holes.  
   *Phys. Rev. D* 95, 044028. [arXiv:1611.03703](https://arxiv.org/abs/1611.03703)

6. **Nagar, A. et al. (2018)** — Time-domain effective-one-body gravitational waveforms for coalescing compact binaries with nonprecessing spins, tides, and self-spin effects.  
   *Phys. Rev. D* 98, 104052. [arXiv:1806.01772](https://arxiv.org/abs/1806.01772)

#### Phenomenological
7. **Ajith, P. et al. (2007)** — A template bank for gravitational waveforms from coalescing binary black holes.  
   *Phys. Rev. D* 77, 104017. [arXiv:0710.2335](https://arxiv.org/abs/0710.2335)

8. **Husa, S. et al. (2016)** — Frequency-domain gravitational waves from nonprecessing black-hole binaries. I. New numerical waveforms and anatomy of the signal (IMRPhenomD).  
   *Phys. Rev. D* 93, 044006. [arXiv:1508.07250](https://arxiv.org/abs/1508.07250)

9. **Hannam, M. et al. (2014)** — Simple Model of Complete Precessing Black-Hole-Binary Gravitational Waveforms (IMRPhenomPv2).  
   *Phys. Rev. Lett.* 113, 151101. [arXiv:1308.3271](https://arxiv.org/abs/1308.3271)

10. **Pratten, G. et al. (2021)** — Computationally efficient models for the dominant and subdominant harmonic modes of precessing binary black holes (IMRPhenomXP).  
    *Phys. Rev. D* 103, 104056. [arXiv:2004.06503](https://arxiv.org/abs/2004.06503)

#### NR Surrogates
11. **Blackman, J. et al. (2017)** — A Surrogate Model of Gravitational Waveforms from Numerical Relativity Simulations of Precessing Binary Black Hole Mergers.  
    *Phys. Rev. D* 96, 024058. [arXiv:1705.07089](https://arxiv.org/abs/1705.07089)

12. **Varma, V. et al. (2019)** — Surrogate model of gravitational waveforms from numerical relativity simulations of precessing binary black holes over a large parameter space (NRSur7dq4).  
    *Phys. Rev. D* 99, 064045. [arXiv:1905.09300](https://arxiv.org/abs/1905.09300)

13. **Varma, V. et al. (2019)** — High-accuracy waveforms for binary black hole mergers with nearly extremal spins (NRHybSur3dq8).  
    *Phys. Rev. Research* 1, 033015. [arXiv:1904.08462](https://arxiv.org/abs/1904.08462)

### Physical Effects
14. **Apostolatos, T. A., Cutler, C., Sussman, G. J. & Thorne, K. S. (1994)** — Spin-induced orbital precession and its modulation of the gravitational waveforms from merging binaries.  
    *Phys. Rev. D* 49, 6274.

15. **Peters, P. C. & Mathews, J. (1963)** — Gravitational Radiation from Point Masses in a Keplerian Orbit.  
    *Phys. Rev.* 131, 435.

16. **Flanagan, É. É. & Hinderer, T. (2008)** — Measuring neutron-star tidal deformability with gravitational waves.  
    *Phys. Rev. D* 77, 021502(R). [arXiv:0709.1915](https://arxiv.org/abs/0709.1915)

17. **Abbott et al. (2017)** — GW170817: Observation of Gravitational Waves from a Binary Neutron Star Inspiral.  
    *Phys. Rev. Lett.* 119, 161101. [arXiv:1710.05832](https://arxiv.org/abs/1710.05832)

### Mismatch and Template Banks
18. **Owen, B. J. (1996)** — Search templates for gravitational waves from inspiraling binaries.  
    *Phys. Rev. D* 53, 6749. [arXiv:gr-qc/9511032](https://arxiv.org/abs/gr-qc/9511032)

19. **Damour, T., Iyer, B. R. & Sathyaprakash, B. S. (1998)** — Improved filters for gravitational waves from inspiraling compact binaries.  
    *Phys. Rev. D* 57, 885.

### Software and Tools
20. **PyCBC** — [https://pycbc.org](https://pycbc.org)  
    Nitz, A. et al. (2024). *gwastro/pycbc*. [GitHub](https://github.com/gwastro/pycbc)

21. **LALSuite** — LSC Algorithm Library.  
    [https://git.ligo.org/lscsoft/lalsuite](https://git.ligo.org/lscsoft/lalsuite)

22. **GWpy** — [https://gwpy.github.io](https://gwpy.github.io)

23. **gwsurrogate** — [https://github.com/sxs-collaboration/gwsurrogate](https://github.com/sxs-collaboration/gwsurrogate)

24. **Bilby** — Bayesian inference library for GW astronomy.  
    Ashton, G. et al. (2019). *ApJS* 241, 27. [arXiv:1811.02042](https://arxiv.org/abs/1811.02042)

### Review Articles and Books
25. **Maggiore, M. (2007)** — *Gravitational Waves: Theory and Experiments*, Vol. 1. Oxford University Press.

26. **Sathyaprakash, B. S. & Schutz, B. F. (2009)** — Physics, Astrophysics and Cosmology with Gravitational Waves.  
    *Living Rev. Rel.* 12, 2. [arXiv:0903.0338](https://arxiv.org/abs/0903.0338)

27. **Hannam, M. (2009)** — Modelling gravitational waves from precessing black-hole binaries.  
    *Class. Quantum Grav.* 26, 114001.

28. **Cotesta, R. et al. (2018)** — Enriching the symphony of gravitational waves from binary black holes by tuning higher harmonics.  
    *Phys. Rev. D* 98, 084028. [arXiv:1803.10701](https://arxiv.org/abs/1803.10701)

---

*Lesson 7 — Gravitational-Wave Waveform Models*  
*LVK Python Course — Module 7*  
*Last updated: 2026*
